# Batch lattice-parameter calculation

Install the repository requirements plus `pandas` in a Python 3 Jupyter kernel, then run all cells. This notebook reads the Cu Kα wavelengths and doublet ratio once from the ST TXT header. Every sample profile, including ST, comes from a V4 CSV. Use the same settings for scans measured with the same instrument configuration.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
from scipy.signal import find_peaks

current = Path.cwd()
root = next(
    (path for path in (current, *current.parents) if (path / "05_batch_v4_outputs").is_dir()),
    None,
)
if root is None:
    raise FileNotFoundError("Start Jupyter from the repository or one of its subfolders.")

reference_txt = root / "01_original_data" / "XRD_Al6063_ST_2015_06_08_ASCII.txt"
input_dir = root / "05_batch_v4_outputs"
output_dir = root / "07_lattice_parameter_results"

def read_wavelengths(path):
    header = []
    with path.open(encoding="utf-8-sig") as scan:
        for line in scan:
            if line.strip() == "[Data]":
                break
            header.append(line)
    metadata = dict(re.findall(r"(?m)^([A-Za-z][A-Za-z0-9]*)=([^\r\n]*)", "".join(header)))
    return float(metadata["Alpha1"]), float(metadata["Alpha2"]), float(metadata["AlphaRatio"])

wavelength, alpha2, alpha_ratio = read_wavelengths(reference_txt)

## Fit the FCC reflections

A Gaussian Cu Kα doublet with a linear background estimates the Kα₁ peak centres. The lattice parameter uses (200), (220) and (311) for every sample. The (111) fit is reported when detected.

In [2]:
# Approximate 2θ search centres for the FCC aluminium reflections.
reflections = {3: ("(111)", 38.5), 4: ("(200)", 44.7), 8: ("(220)", 65.1), 11: ("(311)", 78.2)}
common = (4, 8, 11)

def doublet(x, baseline, slope, amplitude, center, sigma):
    center2 = 2 * np.rad2deg(np.arcsin(alpha2 / wavelength * np.sin(np.deg2rad(center / 2))))
    first = np.exp(-0.5 * ((x - center) / sigma) ** 2)
    second = np.exp(-0.5 * ((x - center2) / sigma) ** 2)
    return baseline + slope * (x - center) + amplitude * (first + alpha_ratio * second)

def fit_peaks(x, y, sample):
    if len(x) < 3 or not np.all(np.isfinite([x, y])) or not np.all(np.diff(x) > 0):
        raise ValueError(f"{sample}: expected finite intensities and strictly increasing 2θ.")

    smoothed = gaussian_filter1d(y, sigma=1)
    candidates, info = find_peaks(smoothed, prominence=0.01 * np.ptp(smoothed))
    fits = []
    for n, (hkl, expected) in reflections.items():
        nearby = np.flatnonzero(np.abs(x[candidates] - expected) <= 0.55)
        if not nearby.size:
            if n in common:
                raise ValueError(f"{sample}: missing required FCC peak {hkl}.")
            continue

        index = candidates[nearby[np.argmax(info["prominences"][nearby])]]
        seed = x[index]
        half_width = 0.75 if n == 11 else 0.55
        window = (x >= seed - half_width) & (x <= seed + half_width)
        xx, yy = x[window], y[window]
        if len(xx) < 7:
            raise ValueError(f"{sample}: too few points to fit peak {hkl}.")

        base = max(0, float(np.median(np.r_[yy[:3], yy[-3:]])))
        height = float(yy.max())
        params, _ = curve_fit(
            doublet, xx, yy,
            p0=[base, 0, max(1, height - base), seed, 0.11],
            bounds=(
                [0, -1000, 0.01, seed - 0.25, 0.035],
                [max(2 * height, 2), 1000, max(3 * height, 3), seed + 0.25, 0.6],
            ),
            maxfev=30000,
        )
        rmse = np.sqrt(np.mean((yy - doublet(xx, *params)) ** 2))
        fits.append({"N": n, "hkl": hkl, "two_theta": params[3],
                     "fit_sigma": params[4], "fit_RMSE": rmse, "Sample": sample})
    return pd.DataFrame(fits)

## Calculate and consolidate

Convert peak centres to apparent lattice parameters using Bragg's law. Calculate their arithmetic mean, the Nelson–Riley intercept, and a Cohen-style least-squares estimate with a Nelson–Riley correction term.

In [3]:
methods = ["Arithmetic mean (Å)", "Nelson-Riley (Å)", "Cohen LS + N-R (Å)"]

def calculate_lattice(peaks):
    selected = peaks.loc[peaks["N"].isin(common)].copy()
    theta = np.deg2rad(selected["two_theta"].to_numpy() / 2)
    n = selected["N"].to_numpy()
    sin_theta = np.sin(theta)
    selected["theta_rad"] = theta
    selected["d_A"] = wavelength / (2 * sin_theta)
    selected["a_A"] = selected["d_A"] * np.sqrt(n)
    f_nr = 0.5 * np.cos(theta)**2 * (1 / sin_theta + 1 / theta)
    selected["F_NR"] = f_nr

    a_mean = selected["a_A"].mean()
    a_nr = np.polyfit(f_nr, selected["a_A"], 1)[1]
    design = np.column_stack((n, n * f_nr))
    coefficient = np.linalg.lstsq(design, sin_theta**2, rcond=None)[0][0]
    if coefficient <= 0:
        raise ValueError("The Cohen-style fit returned a nonphysical coefficient.")
    a_cohen = wavelength / (2 * np.sqrt(coefficient))
    return selected[[col for col in selected if col != "Sample"] + ["Sample"]], dict(zip(methods, (a_mean, a_nr, a_cohen)))

In [4]:
def run_batch():
    profiles = sorted(input_dir.glob("XRD_*_reconstructed_v4.csv"))
    if not profiles:
        raise FileNotFoundError(f"No reconstructed XRD profiles in {input_dir}.")

    fitted, indexed, estimates = [], [], []
    for profile in profiles:
        sample = profile.stem.removeprefix("XRD_").removesuffix("_reconstructed_v4").removesuffix("_2015")
        data = pd.read_csv(profile, usecols=["angle_degrees", "intensity_estimated"])
        x = data["angle_degrees"].to_numpy(dtype=float)
        y = data["intensity_estimated"].to_numpy(dtype=float)
        peaks = fit_peaks(x, y, sample)
        used, results = calculate_lattice(peaks)
        fitted.append(peaks)
        indexed.append(used)
        estimates.append({"Sample": sample, **results})

    summary = pd.DataFrame(estimates).set_index("Sample")
    if "ST" in summary.index:
        summary = summary.loc[["ST", *summary.index.drop("ST")]]

    output_dir.mkdir(exist_ok=True)
    pd.concat(fitted, ignore_index=True).to_csv(output_dir / "all_samples_fitted_peaks.csv", index=False)
    pd.concat(indexed, ignore_index=True).to_csv(output_dir / "all_samples_indexed_bragg.csv", index=False)
    summary.to_csv(output_dir / "all_samples_lattice_parameters.csv")
    return summary

In [5]:
print(f"Cu Kα₁ wavelength: {wavelength:.4f} Å")
display(run_batch().round(6))

Cu Kα₁ wavelength: 1.5406 Å


,Arithmetic mean (Å),Nelson-Riley (Å),Cohen LS + N-R (Å)
Sample,,,
ST,4.054361,4.053861,4.053963
CR40,4.052420,4.052331,4.052622
CR40_annealed,4.053312,4.052320,4.052584
UDR85,4.051311,4.050562,4.051404
UDR85_annealed,4.054511,4.053298,4.053074
